# 01 · Evaluación Traditional RAG
Evalúa el Traditional RAG (ChromaDB + MiniLM + Gemini) sobre UltraDomain con métricas RAGAS.

In [1]:
import os
import sys
from dotenv import load_dotenv

sys.path.append(os.path.abspath('..'))
sys.path.append(os.path.abspath('../..'))
os.listdir()

['01_eval_traditional.ipynb', '00_DatasetLoading.ipynb']

## 1. Configuración del experimento

In [2]:
DOMINIO     = "cs"   # dominio de UltraDomain
N_LIBROS    = 2      # cuántos libros indexar
MAX_Q       = None      # preguntas a evaluar (None = todas)
SHUFFLE     = False  # True = libros aleatorios
CARPETA_DB  = "../../chroma_db_eval_traditional"
RESULTS_DIR = "../../results_traditionalRAG/"

## 2. Cargar datos

In [3]:
from src.ultradomain import cargar_experimento

libros, qas = cargar_experimento(DOMINIO, n_libros=N_LIBROS, shuffle=SHUFFLE)

📚 Dominio: cs
   📖 Guide to Java — James T. Streib (8 preguntas)
   📖 Machine Learning With Spark — Nick Pentreath (8 preguntas)
   Total Q&A: 16


In [4]:
# En vez de cargar el primer libro, cargar el que corresponde a las Q&A
# DOMINIO  = "cs"
# N_LIBROS = 1
# SHUFFLE  = False

# libros, qas = cargar_experimento(DOMINIO, n_libros=N_LIBROS, shuffle=SHUFFLE)

# Verificar que coinciden
print(f"Libro indexado: {libros[0]['titulo']}")
print(f"Q&A son sobre: {qas[0]['titulo']}")

Libro indexado: Guide to Java
Q&A son sobre: Machine Learning With Spark


## 3. Inicializar y indexar el RAG

In [5]:
import shutil
from src.data_loader import load_and_split_text
from src.baselines.traditional_rag import TraditionalRAG

# Empezar limpio
shutil.rmtree(CARPETA_DB, ignore_errors=True)

rag = TraditionalRAG(persist_directory=CARPETA_DB)

for libro in libros:
    # Guardar texto en fichero temporal para load_and_split_text
    tmp = f"/tmp/{libro['context_id']}.txt"
    with open(tmp, "w", encoding="utf-8") as f:
        f.write(libro["texto"])
    splits = load_and_split_text(tmp)
    print(f"📖 {libro['titulo']} → {len(splits)} fragmentos")
    rag.index_documents(splits)

print("✅ Indexación completada")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


📖 Guide to Java → 845 fragmentos
📖 Machine Learning With Spark → 764 fragmentos
✅ Indexación completada


## 4. Ejecutar evaluación

In [6]:
from src.evaluation.experiment import run_experiment 

result = await run_experiment(
    rag_type="traditional",
    rag_object=rag,
    dominio=DOMINIO,
    n_libros=N_LIBROS,
    max_questions=MAX_Q,
    shuffle=SHUFFLE,
    save_path=RESULTS_DIR,
)

/Volumes/Toni 1TB/MESIIA/TFM/TFM_Repositori/Code_TFM/venv/lib/python3.10/site-packages/google/api_core/_python_version_support.py:275: FutureWarning: You are using a Python version (3.10.20) which Google will stop supporting in new releases of google.api_core once it reaches its end of life (2026-10-04). Please upgrade to the latest Python version, or at least Python 3.11, to continue receiving updates for google.api_core past that date.
  warnings.warn(message, FutureWarning)
/Volumes/Toni 1TB/MESIIA/TFM/TFM_Repositori/Code_TFM/venv/lib/python3.10/site-packages/instructor/providers/gemini/client.py:6: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  import google.generativeai as genai


📚 Dominio: cs
   📖 Guide to Java — James T. Streib (8 preguntas)
   📖 Machine Learning With Spark — Nick Pentreath (8 preguntas)
   Total Q&A: 16

🔍 Evaluando TRADITIONAL | cs | 16 preguntas
────────────────────────────────────────────────────────────
  [1/16] How does Spark Streaming enable real-time data processing?...
  [2/16] What are the key features of this text that aid in learning object-ori...
  [3/16] How does the text compare to other Java programming texts in terms of ...
  [4/16] What audience is the text primarily intended for?...
  [5/16] What is the significance of the ALS algorithm in Spark's MLlib?...
  [6/16] What is Apache Spark and what are its key features?...
  [7/16] How does the MovieLens dataset contribute to building recommendation e...
  [8/16] What tools or methodologies does the text use to help readers understa...
  [9/16] What role do examples and exercises play in the learning process accor...
  [10/16] What is the primary purpose of the text "Guide to 

## 5. Inspeccionar respuestas individuales

In [7]:
for r in result.qa_results:
    print(f"❓ {r.question}")
    print(f"✅ GT:  {r.ground_truth[:150]}...")
    print(f"🤖 RAG: {r.answer[:150]}...")
    print(f"⏱️  {r.latency_s}s\n")

❓ What is the significance of the R tool in the context of modern optimization methods?
✅ GT:  The R tool is significant because it is a free, open-source, and multi-platform tool specifically developed for statistical analysis, and it has an ac...
🤖 RAG: The R tool is significant in the context of modern optimization methods for several reasons:

1.  **Practical Implementation:** It serves as a practic...
⏱️  4.95s

❓ What are the three main approaches to handle multi-objective tasks discussed in the book?
✅ GT:  The three main approaches discussed are the weighted-formula approach, lexicographic approach, and Pareto front approach....
🤖 RAG: The three main approaches to handle multi-objective tasks discussed in the book are: weighted-formula, lexicographic, and Pareto front....
⏱️  1.43s

❓ Can you name some popular modern optimization methods discussed in the book?
✅ GT:  Some popular modern optimization methods discussed in the book include simulated annealing, tabu search, genetic